# 实验 4 — dbt lineage & docs

**目标**：理解 dbt 的依赖图（DAG），用代码查看模型间依赖关系，再用 docs serve 可视化。

## 4a. 用代码读 dbt manifest（不开浏览器也能看依赖）

In [ ]:
import json
from pathlib import Path

manifest_path = Path("../dbt_project/target/manifest.json")

if not manifest_path.exists():
    print("manifest.json 不存在，先运行 make build 生成")
else:
    manifest = json.loads(manifest_path.read_text())
    nodes = manifest["nodes"]
    
    print("所有 model 节点及其直接依赖：")
    for key, node in nodes.items():
        if node["resource_type"] == "model":
            deps = node["depends_on"]["nodes"]
            print(f"  {node['name']}")
            for dep in deps:
                short = dep.split(".")[-1]
                print(f"    ← {short}")

In [ ]:
# 看 sources 的定义
if manifest_path.exists():
    sources = manifest.get("sources", {})
    print("Sources：")
    for key, src in sources.items():
        print(f"  {src['source_name']}.{src['name']} → {src['schema']}.{src['name']}")

## 4b. 启动 dbt docs server（在终端里跑）

```bash
make docs
```

然后打开 http://localhost:8080，点击右下角的图标进入 lineage graph。

**导航提示**：
- 点击 `fct_daily_rates`，右侧会显示 "Parents" 和 "Children"
- 点击 "See lineage graph" 可看完整 DAG：`raw_ecb.daily_rates` → `stg_ecb_rates` → `fct_daily_rates`

## 4c. 用代码查看 catalog（物化后的实际列）

In [ ]:
catalog_path = Path("../dbt_project/target/catalog.json")

if not catalog_path.exists():
    print("catalog.json 不存在，先运行 make docs（会先 dbt docs generate）")
else:
    catalog = json.loads(catalog_path.read_text())
    for node_key, node_data in catalog["nodes"].items():
        name = node_data["metadata"]["name"]
        cols = list(node_data["columns"].keys())
        print(f"  {name}: {cols}")